### Imports 


In [2]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import pickle

### Create a Regression dataset
features: study hours, sleep hours, practice tests
target: Exam score

In [3]:
def create_regression_dataset(n_samples=250, random_state=42):
    """
    random_state: keeps the random numbers same every time we ran the notebook,
    this will helps us to get the same results every time we run the notebook.

    by setting randomstate = 42, 
    we are making sure that the random numbers generated are same every time we run the notebook.
    """
    rng = np.random.default_rng(random_state)

    # create study column realistic ranges.

    study_hours = rng.uniform(1, 10, n_samples)
    sleep_hours = rng.uniform(4, 9, n_samples)
    practice_tests = rng.uniform(0, 8, n_samples)

    # combine the 3 features columns into one 2d matrix
    X = np.column_stack((study_hours, sleep_hours, practice_tests))

    # Add noise to the target variable to get realistic results
    noise = rng.normal(0, 5, n_samples)


    # the formula to create exam scores,
    #  f(x) = base_score + (study_hours * 5) + (sleep_hours * 3) + (practice_tests * 2) + noise

    y = 20 + (5 * study_hours) + (3 * sleep_hours) + (2 * practice_tests) + noise

    return X, y

# Create the dataset
X, y = create_regression_dataset() # sores the returned values from the function into X and y

# show dataset shape
print("Feature matrix shape: ", X.shape )
print("Target vector shape: ", y.shape )

# show the first 10 rows of the feature matrix
print("\nFirst 10 rows of the feature matrix:\n", X[:10])

# show the first 10 values of the target vector
print("\nFirst 10 values of the target vector:\n", y[:10])


Feature matrix shape:  (250, 3)
Target vector shape:  (250,)

First 10 rows of the feature matrix:
 [[7.96560444 7.25965513 5.89364551]
 [4.94990596 8.33745316 7.09122309]
 [8.72738128 6.26948441 7.36845758]
 [7.27631226 5.23919781 4.0290634 ]
 [1.84759613 5.18331181 4.16220092]
 [9.78060116 7.7300714  6.39896329]
 [7.85025732 8.08284382 2.51560553]
 [8.07457875 4.5263904  6.6990589 ]
 [2.15302269 4.33279428 3.95313317]
 [5.05347344 6.97216832 0.92685379]]

First 10 values of the target vector:
 [ 92.9133839   89.58442974  85.77858556  72.67408811  48.49788573
 112.19704093  89.94396402  91.18676894  45.96895807  62.43990037]


In [4]:
def train_test_split_from_scratch(X, y, test_size=0.2, random_state=42):
    """"
    splits the dataset into training and testing sets without sklearn.

    test_size = 0.20 means 20% of the dataset will be used for testing and 80% for training.
    """

    rng = np.random.default_rng(random_state)

    # Create an array of indices for all data points and assign row numbers for each data. (0-249)
    incices = np.arange(len(X))

    # shuffle indexes to randomly select data points for training and testing sets
    rng.shuffle(incices)

    # calculate the number of samples for the test set
    test_size = int(len(X) * test_size)

    # The dataset is already shuffled, so the first part of the shuffled indices will be used for the test set,
    #  and the rest for the training set.
    test_indices = incices[:test_size] # 0-49 randomly selected for testing

    train_indices = incices[test_size:] # 50-249 randomly selected for training


    # split the training and testing sets using the calculated indices
    return X[train_indices], X[test_indices], y[train_indices], y[test_indices]

X_train, X_test, y_train, y_test = train_test_split_from_scratch(X, y)

print("training rows: ", X_train.shape[0])
print("testing rows: ", X_test.shape[0])

training rows:  200
testing rows:  50


# Standardize the data

Rescale and transfrom the features of dataset to have a mean of 0 and a standard deviation of 1.
this means (X - mean) => transformed data have mean of ZERO

$Standardized Values = \frac{x - \mu}{\sigma}$

In [5]:
def standardize_train_test(X_train, X_test):
    """
    we calculate the mean and standard deviation of the training data only,
    This will avoid data leakage from the test set into the training set.
    """

    # Calulate the mean value per column
    mean = np.mean(X_train, axis=0)

    # Calculate the standard deviation per column
    std = np.std(X_train, axis=0)

    # Avoid division by zero if a column has zero standard deviation ( something / 0 = ininity)
    std[std == 0] = 1

    # apply the standardization formula to both training and testing sets
    X_train_scaled = (X_train - mean) / std
    X_test_scaled = (X_test - mean) / std

    return X_train_scaled, X_test_scaled

X_train_scaled, X_test_scaled = standardize_train_test(X_train, X_test)

print("First 10 rows of the standardized training set:\n", X_train_scaled[:10])

print("First 10 rows of the standardized test set:\n", X_test_scaled[:10])



First 10 rows of the standardized training set:
 [[ 1.02961789  0.64788709 -1.39841404]
 [-0.57046859  1.54200536  0.89007783]
 [ 0.62267758  0.83754901 -0.11108981]
 [-1.24690275  0.9999835   0.47561088]
 [ 0.74894291 -1.13543923  0.78057951]
 [ 1.43063277  1.51283761 -1.57668939]
 [-1.31260323  1.00973958 -0.3162726 ]
 [-0.0501766  -0.42717227 -1.24875169]
 [-0.22791771  0.05985983  1.05142967]
 [ 0.05907697  0.49278543 -0.1876875 ]]
First 10 rows of the standardized test set:
 [[ 0.33651153  0.93430418  0.83460128]
 [-1.0340711   0.99194006 -0.21121848]
 [ 0.76841499  1.26700854 -1.65314423]
 [-0.3649215   1.69408605 -0.47182516]
 [ 1.34839014  0.18730247 -0.14271724]
 [-1.66588418  1.71880386 -1.43369252]
 [ 0.09021092 -1.60848083  0.14714269]
 [ 0.74059824 -0.86746848 -0.09473797]
 [-1.69751895 -1.41398917 -0.84302338]
 [-0.00423082 -0.0751631  -0.33465417]]


# Model implementation

Linear Regression is a fundamental model in ML used for predicting a continuous output variable based on input variable.
$$f_{w,b} (x) = wx + b $$

## cost function

The cost function is used to measure how well the model is performing. It quantifies the difference between the predicted and actual values in our dataset.
$J(w,b) = \frac{1}{n} \sum_{i=1}^{n} (y^{(i)} - y^{-})^2$

In [7]:
class LinearRegression:
    """
    Linear Regression model from scratch using gradient descent.
    """

    def __init__(self, learning_rate=0.01, n_iterations=1000):

        self.learning_rate = learning_rate
        self.n_iterations = n_iterations

        # Initialize weights and bias to None, they will be set during training
        self.weights = None
        self.bias = None

        # This list stores the loss values for each iteration
        self.loss_history = []

    def fit(self, X, y):
        """
        Train the model using gradient descent.
        """
        n_samples, n_features = X.shape

        # Initialize weights and bias to zeros
        self.weights = np.zeros(n_features)
        self.bias = 0

        for _ in range(self.n_iterations):
            #make predictions using the current weights and bias
            y_predicted = self.predict(X)

            # Calculate the gradients for weights and bias

            # X.T is the transpose of the feature matrix X,
            #  which allows us to compute the dot product with the error term (y_predicted - y) 
            # to get the gradient for weights.
            dw = (1 / n_samples) * np.dot(X.T, (y_predicted - y))
            db = (1 / n_samples) * np.sum(y_predicted - y)

            # Update the weights and bias using the gradients and learning rate
            # Move weights and bias in the direction that lowers the error

            self.weights = self.weights - self.learning_rate * dw
            self.bias = self.bias - self.learning_rate * db

            # Stores the loss so we can check if training is improving
            loss = self.mean_squared_error(y, y_predicted)
            self.loss_history.append(loss)

    def predict(self, X):
        """
        Predict the target values using the learned weights and bias.
        """
        return np.dot(X, self.weights) + self.bias

    def mean_squared_error(self, y_true, y_pred):
        """
        Calculate the mean squared error between the true and predicted values.
        """
        return np.mean((y_true - y_pred) ** 2)

    def r2_score(self, y_true, y_pred):
        """
        Calculate the R-squared score to evaluate the model's performance.
        Best value is 1.0
        A value near 0 means the model is not better than predicting the mean of the target variable.
        """
        total_variation = np.sum((y_true - np.mean(y_true)) ** 2)
        unexplained_variation = np.sum((y_true - y_pred) ** 2)
        return 1 - (unexplained_variation / total_variation)



# train the model
Now we are creating the model and call fit() on the training data

In [13]:
#create the model
model = LinearRegression(learning_rate=0.01, n_iterations=1000)

# Train the model using scaled training data.
model.fit(X_train_scaled, y_train)

print("training complete!")
print("Final weights: ", model.weights)
print("Final bias: ", model.bias)

training complete!
Final weights:  [12.45738487  4.37765192  5.46870628]
Final bias:  74.7703303653321


# Make predictions

We use the test data to check how well the model works on unseen examples.

In [14]:
predictions = model.predict(X_test_scaled)


# Show first 10 predictions and their corresponding actual values
for predicted, actual in zip(predictions[:10], y_test[:10]):
    print(f"Predicted: {predicted:.2f}, Actual: {actual:.2f}")

Predicted: 87.62, Actual: 96.69
Predicted: 65.08, Actual: 73.84
Predicted: 80.85, Actual: 82.37
Predicted: 75.06, Actual: 75.83
Predicted: 91.61, Actual: 85.29
Predicted: 53.70, Actual: 54.64
Predicted: 69.66, Actual: 67.13
Predicted: 79.68, Actual: 72.67
Predicted: 42.82, Actual: 42.70
Predicted: 72.56, Actual: 77.46


# Evaluate the model

We will calculate:
  ### Mean squared error, R2 score

In [15]:
mse = model.mean_squared_error(y_test, predictions)
r2 = model.r2_score(y_test, predictions)

print(f"Test MSE: {mse:.4f}")
print(f"Test R2 score: {r2:.4f}")

Test MSE: 23.2878
Test R2 score: 0.8850


# Inspect the loss

The loss should go down during training
If the loss goes down, gradient descent is learning

In [16]:
print("First 10 loss values: ")
print(model.loss_history[:10])

print("\nLast 10 loss values: ")
print(model.loss_history[-10:])

First 10 loss values: 
[np.float64(5810.908282993585), np.float64(5696.015581590303), np.float64(5583.405768801301), np.float64(5473.033474329496), np.float64(5364.854229752773), np.float64(5258.82445059306), np.float64(5154.901418741945), np.float64(5053.0432652357695), np.float64(4953.208953373249), np.float64(4855.358262168799)]

Last 10 loss values: 
[np.float64(27.249650859420225), np.float64(27.249650565645858), np.float64(27.24965027764006), np.float64(27.249649995289403), np.float64(27.249649718482704), np.float64(27.249649447110997), np.float64(27.249649181067408), np.float64(27.249648920247182), np.float64(27.249648664547657), np.float64(27.24964841386816)]
